<a href="https://colab.research.google.com/github/weagan/Speculative-Decoding/blob/main/draft_cpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Draft Notebook — TinyLlama + remote Gemma-2B (speculative decoding)

Runs TinyLlama on **CPU (or GPU if available)** as the draft model, sends speculative
token sequences to the target GPU notebook via ngrok, and measures effective throughput.

Works on **Colab** and **Kaggle** with 0, 1, or 2 GPUs.

In [ ]:
import os, torch

def detect_env():
    if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
        return 'colab'
    if os.path.exists('/kaggle'):
        return 'kaggle'
    return 'local'

ENV = detect_env()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_COUNT = torch.cuda.device_count()

print(f'Environment : {ENV}')
print(f'Device      : {DEVICE}  ({GPU_COUNT} GPU(s) visible)')
if GPU_COUNT:
    for i in range(GPU_COUNT):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name}  {props.total_memory/1e9:.1f} GB')


Environment : colab
Device      : cpu  (0 GPU(s) visible)


In [ ]:
# ── Load secrets (Colab: Settings > User data | Kaggle: Add-ons > Secrets) ──
HF_TOKEN = None

if ENV == 'colab':
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
elif ENV == 'kaggle':
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
else:
    HF_TOKEN = os.getenv('HF_TOKEN')

assert HF_TOKEN, 'HF_TOKEN not set — add it to your environment secrets.'
print('HF token loaded.')

HF token loaded.


In [ ]:
!pip install -q transformers requests
print('Dependencies installed.')


Dependencies installed.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

DRAFT_MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
print(f'Loading draft model: {DRAFT_MODEL_ID} ...')

# Use float16 on GPU, float32 on CPU
dtype = torch.float16 if DEVICE == 'cuda' else torch.float32

draft_tokenizer = AutoTokenizer.from_pretrained(DRAFT_MODEL_ID, token=HF_TOKEN)
draft_model = AutoModelForCausalLM.from_pretrained(
    DRAFT_MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=dtype,
    device_map='auto' if DEVICE == 'cuda' else None,
)
if DEVICE == 'cpu':
    draft_model = draft_model.to('cpu')
draft_model.eval()
print(f'Draft model loaded. Running on: {next(draft_model.parameters()).device}')


Loading draft model: TinyLlama/TinyLlama-1.1B-Chat-v1.0 ...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Draft model loaded. Running on: cpu


In [ ]:
import requests

# ── Paste the ngrok URL printed by the target notebook here ──────────────────
TARGET_BASE_URL = "https://fd9b-35-187-144-146.ngrok-free.app"  # ← replace this
assert TARGET_BASE_URL and "xxxx" not in TARGET_BASE_URL, \
    "Set TARGET_BASE_URL to the ngrok URL printed by the target notebook."

# Quick connectivity check
try:
    r = requests.get(TARGET_BASE_URL + "/", timeout=10)
    print("Target reachable:", r.json())
except Exception as e:
    raise RuntimeError(f"Cannot reach target at {TARGET_BASE_URL}: {e}")


def verify_with_target(context_text: str, draft_text: str) -> dict:
    """
    Send context + draft continuation (as text strings) to the target.
    Returns dict with keys: draft_text, target_text, elapsed, tokens_per_sec.

    Why text and not token IDs?
    Draft (TinyLlama) and target (Gemma-2B) have DIFFERENT vocabularies.
    Token ID 42 in TinyLlama is a completely different word than ID 42 in Gemma.
    Sending raw IDs across models produces garbage. Text is the shared language.
    """
    payload = {"context_text": context_text, "draft_text": draft_text}

    print(f"  [PAYLOAD →] context_text (last 80): {repr(context_text[-80:])}")
    print(f"  [PAYLOAD →] draft_text            : {repr(draft_text)}")

    r = requests.post(TARGET_BASE_URL + "/verify", json=payload, timeout=60)
    r.raise_for_status()
    data = r.json()

    print(f"  [PAYLOAD ←] draft_text  : {repr(data['draft_text'])}")
    print(f"  [PAYLOAD ←] target_text : {repr(data['target_text'])}")

    return data


print("verify_with_target ready.")


Target reachable: {'status': 'ok', 'model': 'gemma-2b-it'}
verify_with_target ready.


In [ ]:
import time

def format_prompt(text: str) -> str:
    """Wrap plain text in TinyLlama ChatML format."""
    messages = [{"role": "user", "content": text}]
    return draft_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def speculative_decode(prompt: str, max_new_tokens: int = 64, draft_len: int = 8) -> tuple:
    """
    Distributed speculative decoding — prefix-match strategy over TEXT.

    All cross-notebook communication happens as decoded text strings,
    never raw token IDs, because draft and target use different tokenizers
    (TinyLlama vocab ≠ Gemma vocab).

    Each round:
      1. Draft greedily generates `draft_len` tokens → decode to draft_text.
      2. Send (context_text, draft_text) to target → receive target_text.
      3. Split both into words and find the longest matching prefix.
      4a. prefix > 0: accept prefix + target's correction word.
      4b. prefix = 0: accept target's first word.
      5. Re-encode accepted text with draft tokenizer, extend context, repeat.
    """
    formatted   = format_prompt(prompt)
    id_list     = draft_tokenizer(formatted, return_tensors="pt").input_ids[0].tolist()
    context_text = formatted          # running text context sent to target
    accepted_text = ""                # accumulates newly generated text
    total_new   = 0
    step        = 0
    t0          = time.perf_counter()

    while total_new < max_new_tokens:
        step += 1
        print(f"\n{'─'*60}")
        print(f"  STEP {step}  |  context tokens: {len(id_list)}  |  new so far: {total_new}")

        # ── 1. Draft: greedily generate draft_len tokens ──────────────────────
        draft_ids = []
        cur = torch.tensor([id_list], dtype=torch.long).to(DEVICE)
        with torch.no_grad():
            for _ in range(draft_len):
                next_id = draft_model(cur).logits[:, -1, :].argmax(dim=-1).item()
                draft_ids.append(next_id)
                cur = torch.cat([cur, torch.tensor([[next_id]], device=DEVICE)], dim=1)
                if next_id == draft_tokenizer.eos_token_id:
                    break

        if not draft_ids:
            print("  [DRAFT] Generated nothing — stopping.")
            break

        draft_text = draft_tokenizer.decode(draft_ids, skip_special_tokens=True)
        print(f"  [DRAFT] ids  : {draft_ids}")
        print(f"  [DRAFT] text : {repr(draft_text)}")

        # ── 2. Target: verify over text ────────────────────────────────────────
        result      = verify_with_target(context_text, draft_text)
        target_text = result["target_text"]
        tgt_elapsed = result.get("elapsed")
        tgt_tps     = result.get("tokens_per_sec")
        tgt_str     = f"{tgt_elapsed:.3f}s  {tgt_tps:.1f} tok/s" if tgt_elapsed else "n/a"
        print(f"  [TARGET] text : {repr(target_text)}  [{tgt_str}]")

        # ── 3. Find longest matching prefix (word-level) ───────────────────────
        # Split on whitespace for a simple, robust comparison.
        draft_words  = draft_text.split()
        target_words = target_text.split()

        prefix_len = 0
        for dw, tw in zip(draft_words, target_words):
            if dw == tw:
                prefix_len += 1
            else:
                break

        # Build a visual comparison
        comparison = []
        for i, (dw, tw) in enumerate(zip(draft_words, target_words)):
            if i < prefix_len:
                comparison.append(f"✓'{dw}'")
            elif i == prefix_len:
                comparison.append(f"✗[d='{dw}' t='{tw}']")
            else:
                comparison.append(f"  [d='{dw}' t='{tw}']")
        print(f"  [MATCH] {'  '.join(comparison)}")
        print(f"  [MATCH] prefix words = {prefix_len} / {len(draft_words)}")

        # ── 4. Extend context ─────────────────────────────────────────────────
        if prefix_len > 0:
            accepted_words = draft_words[:prefix_len]
            correction     = [target_words[prefix_len]] if prefix_len < len(target_words) else []
            new_text       = " ".join(accepted_words + correction)

            print(f"  [ACCEPT]  prefix  : {repr(' '.join(accepted_words))}")
            if correction:
                print(f"  [CORRECT] target[{prefix_len}] = {repr(correction[0])}  "
                      f"(replaces draft word {repr(draft_words[prefix_len] if prefix_len < len(draft_words) else '—')})")
        else:
            new_text = target_words[0] if target_words else ""
            print(f"  [MISMATCH] No prefix match — taking target word[0]: {repr(new_text)}")

        # Re-encode the accepted text with draft tokenizer to extend id_list
        if new_text:
            new_ids = draft_tokenizer.encode(" " + new_text, add_special_tokens=False)
            id_list.extend(new_ids)
            total_new     += len(new_ids)
            accepted_text += " " + new_text
            context_text  += " " + new_text   # keep text context in sync

        print(f"  [ADDED]  {repr(new_text)}  ({len(new_ids) if new_text else 0} tokens)")
        print(f"  [TOTAL]  {total_new} new token(s) accepted so far")

        # ── 5. EOS check ──────────────────────────────────────────────────────
        if id_list[-1] == draft_tokenizer.eos_token_id:
            print(f"  [EOS] End of sequence reached.")
            break

    print(f"\n{'═'*60}")
    elapsed = time.perf_counter() - t0
    tps     = total_new / elapsed if elapsed > 0 else float("inf")
    text    = draft_tokenizer.decode(id_list, skip_special_tokens=True)
    print(f"  Done: {total_new} tokens in {elapsed:.2f}s = {tps:.2f} tok/s")
    print(f"{'═'*60}\n")
    return text, tps


print("speculative_decode defined.")


speculative_decode defined.


In [ ]:
PROMPTS = [
    'Explain why transformers use self-attention.',
    'Describe the process of photosynthesis.',
    'What is the difference between RAM and ROM?',
]

SEP = '=' * 55
for i, prompt in enumerate(PROMPTS, 1):
    print(f'\n{SEP}')
    print(f'Prompt {i}: {prompt}')
    print(SEP)
    output, tps = speculative_decode(prompt, max_new_tokens=64, draft_len=8)
    print('\nOutput:')
    print(output)
    print(f'\nThroughput: {tps:.2f} tokens/s')



Prompt 1: Explain why transformers use self-attention.

────────────────────────────────────────────────────────────
  STEP 1  |  context tokens: 28  |  new so far: 0
  [DRAFT] ids  : [13372, 414, 671, 1583, 29899, 1131, 2509, 408]
  [DRAFT] text : 'Transformers use self-attention as'
  [PAYLOAD →] context_text (last 80): '<|user|>\nExplain why transformers use self-attention.</s>\n<|assistant|>\n'
  [PAYLOAD →] draft_text            : 'Transformers use self-attention as'
  [PAYLOAD ←] draft_text  : 'Transformers use self-attention as'
  [PAYLOAD ←] target_text : 'Sure use self-attention for'
  [TARGET] text : 'Sure use self-attention for'  [15.481s  0.4 tok/s]
  [MATCH] ✗[d='Transformers' t='Sure']    [d='use' t='use']    [d='self-attention' t='self-attention']    [d='as' t='for']
  [MATCH] prefix words = 0 / 4
  [MISMATCH] No prefix match — taking target word[0]: 'Sure'
  [ADDED]  'Sure'  (2 tokens)
  [TOTAL]  2 new token(s) accepted so far

─────────────────────────────────────────